## OONX

In [ ]:
import onnx, sys
model_path = "captcha_data/supreme_court/0/model/model_full.pt.onnx"
m = onnx.load(model_path)
print("file:", model_path)
print("ir_version:", m.ir_version)
print("producer_name:", m.producer_name)
print("opset_import:", [(o.domain, o.version) for o in m.opset_import])
print("model size (bytes):", __import__('os').path.getsize(model_path))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ImageClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x: torch.Tensor):
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
torch_model = ImageClassifierModel()
# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
example_inputs = (torch.randn(1, 1, 32, 32),)
onnx_program = torch.onnx.export(torch_model, example_inputs, dynamo=True)

In [ ]:
onnx_program.save("image_classifier_model.onnx")

In [ ]:
import onnx

onnx_model = onnx.load("image_classifier_model.onnx")
onnx.checker.check_model(onnx_model)

In [ ]:
import onnxruntime

onnx_inputs = [tensor.numpy(force=True) for tensor in example_inputs]
print(f"Input length: {len(onnx_inputs)}")
print(f"Sample input: {onnx_inputs}")

ort_session = onnxruntime.InferenceSession(
    "./image_classifier_model.onnx", providers=["CPUExecutionProvider"]
)

onnxruntime_input = {input_arg.name: input_value for input_arg, input_value in zip(ort_session.get_inputs(), onnx_inputs)}

# ONNX Runtime returns a list of outputs
onnxruntime_outputs = ort_session.run(None, onnxruntime_input)[0]

In [ ]:
torch_outputs = torch_model(*example_inputs)

assert len(torch_outputs) == len(onnxruntime_outputs)
for torch_output, onnxruntime_output in zip(torch_outputs, onnxruntime_outputs):
    torch.testing.assert_close(torch_output, torch.tensor(onnxruntime_output))

print("PyTorch and ONNX Runtime output matched!")
print(f"Output length: {len(onnxruntime_outputs)}")
print(f"Sample output: {onnxruntime_outputs}")

## Window application

In [ ]:
import captchaSolver.engine as engine

selected_captchas = ['supreme_court', 'gov24']  # Specify desired captcha IDs here
captcha_pred_models = engine.get_captcha_pred_model_list(captcha_id_list=selected_captchas)
captcha_pred_models

engine.batch_predict_model(model=captcha_pred_models['supreme_court'], pred_image_dir="captcha_data/supreme_court/0/images/draft")

# 대법원 캡차 이미지 수집

`https://ssgo.scourt.go.kr/ssgo/ssgo10l/getCaptchaInf.on` API를 호출하여 캡차 이미지를 다운로드합니다.


In [ ]:
from PIL import Image

image_path = "captcha_data/supreme_court/0/images/draft/1.jpg"
image = Image.open(image_path)

def pred_image_preprocess(image_path: str) -> Image.Image:
    image_size = (120, 40)
    bg_color = 255
    crop = (3, 1, image_size[0]-1, image_size[1]-7)
    image = Image.open(image_path)
    print(f"Original image size: {image.size}, mode: {image.mode}")
    print(f"crop point: {image.size[0]-image_size[0]}, {image.size[1]-image_size[1]}")
    crop_image = image.crop(crop)
    alpha = crop_image.split()[-1] if crop_image.mode == "RGBA" else None
    image = Image.new("L", image_size, bg_color)
    image.paste(crop_image, (1, 1), mask=alpha)
    return image

pred_image = pred_image_preprocess(image_path)
pred_image

In [ ]:
# 필요한 라이브러리 임포트
import requests
from pathlib import Path
from datetime import datetime
import time
from PIL import Image
from io import BytesIO
import base64

url = "https://ssgo.scourt.go.kr/ssgo/ssgo10l/getCaptchaInf.on"
image_dir = "captcha_data/supreme_court/0/images"



In [ ]:

def download_captcha(save_path: Path, total: int, index: int = 0) -> bool:
    # 브라우저에서 호출한 것처럼 보이게 할 헤더
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'ko,en;q=0.9,en-US;q=0.8',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        image_b64 = None
        if isinstance(data, dict):
            if 'data' in data and isinstance(data['data'], dict):
                dma = data['data'].get('dma_captchaInf') or data['data'].get('dmaCaptchaInf')
                if isinstance(dma, dict):
                    image_b64 = dma.get('image')
            if not image_b64:
                image_b64 = data.get('image') or data.get('captchaImage')

        if not image_b64:
            raise ValueError('JSON 응답에 이미지(base64) 필드가 없습니다')

        image_bytes = base64.b64decode(image_b64)

        # 파일명/경로 준비 및 저장
        save_path.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        filename = f"{timestamp}.png"
        filepath = save_path / filename
        with open(filepath, "wb") as f:
            f.write(image_bytes)
        # image.save(filepath, format="PNG")
        # index += 1
        print(f"[{index+1}/{total}] 저장 완료: {filename}")
        return True
    except Exception as e:
        print(f"[{index + 1}/{total}] 오류 발생: {e}")
        return False

In [ ]:
# 이미지 다운로드 실행
import os

total_count = 50  # 다운로드할 이미지 수
success_count = 0
fail_count = 0
save_path = Path(os.path.join(image_dir, "draft"))
DELAY = 0.1  # 요청 간 대기 시간 (초)

for i in range(total_count):
    if download_captcha(save_path, total_count, index=i):
        success_count += 1
    else:
        fail_count += 1
    
    # 마지막 요청이 아니면 대기
    if i < total_count - 1:
        time.sleep(DELAY)

print("-" * 50)
print(f"\n다운로드 완료!")
print(f"성공: {success_count}개")
print(f"실패: {fail_count}개")
print(f"저장 위치: {save_path.absolute()}\n")

In [ ]:
import os
import logging, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import captchaSolver.engine as engine
from captchaSolver.dataclass import TrainData
from captchaSolver.backend.pytorch.core import PyTorchModel

captcha_id = 'supreme_court'
rev = 0

model = engine.get_captcha_model(captcha_id=captcha_id)
train_data: TrainData = model.train_data
train_data.rev = rev
# image_width: int = 200
# image_height: int = 50
engine.batch_predict_model(model=model)
model_path = train_data.get_model_path()
image_path = train_data.choice_pred_image()
pred, confidence = engine.predict(model=model, image_path=image_path)
print("image_path : ", image_path)
print("pred : ", pred)
print("confidence : ", f'{confidence:.4f}')
print("Done!")


In [ ]:
def bg_white(base_dir):
    from pathlib import Path
    from PIL import Image

    # 경로 설정
    source_dir = Path(os.path.join(base_dir, "draft"))
    target_dir = Path(os.path.join(base_dir, "draft"))
    target_dir.mkdir(parents=True, exist_ok=True)

    # 이미지 파일 목록
    image_files = list(source_dir.glob("*.png"))
    print(f"처리할 이미지 개수: {len(image_files)}")

    # 각 이미지 처리: 투명 배경 -> 흰색 배경
    success_count = 0
    for idx, img_path in enumerate(image_files, 1):
        try:
            original_image = Image.open(img_path).convert("RGBA")
            white_background = Image.new("RGBA", original_image.size, (255, 255, 255, 255))
            img = Image.alpha_composite(white_background, original_image)
            img = img.convert("RGB").convert("L")
            target_path = target_dir / img_path.name
            img.save(target_path, format="PNG")

            success_count += 1
            if idx % 50 == 0 or idx == len(image_files):
                print(f"[{idx}/{len(image_files)}] 처리 완료")

        except Exception as e:
            print(f"[{idx}/{len(image_files)}] 오류 발생 ({img_path.name}): {e}")

    print(f"\n작업 완료! 성공: {success_count}/{len(image_files)}")
    print(f"저장 위치: {target_dir.absolute()}")
    
bg_white("captcha_data/supreme_court/0/images")

### 캡차 이미지 인식 및 파일명 변경

학습된 모델을 사용하여 draft 폴더의 이미지를 인식하고, 예측된 레이블로 파일명을 변경합니다.

In [ ]:
import os
from pathlib import Path
from captchaSolver.dataclass import TrainData
import captchaSolver.engine as engine
from captchaSolver.backend.pytorch.core import PyTorchModel

captcha_id = 'dev'
backend = 'pytorch'
rev = 0
draft_image_dir = "captcha_data/dev/0/images/draft"
labeled_image_dir = "captcha_data/dev/0/images/labeled"
Path(labeled_image_dir).mkdir(parents=True, exist_ok=True)

draft_image_files = sorted([str(p) for p in Path(draft_image_dir).glob("*.png")])
print(f"Draft 이미지 개수: {len(draft_image_files)}")

model: PyTorchModel = engine.get_captcha_model(captcha_id=captcha_id)
train_data: TrainData = model.train_data
model.train_data.rev = rev
torch_model: PyTorchModel = model
matched = 0
torch_model.load_prediction_model()

for idx, img_path in enumerate(draft_image_files):
    pred, confidence = engine.predict(model=model, image_path=img_path, verbose=0)
    new_image_path = os.path.join(labeled_image_dir, pred + ".png")
    if(os.path.exists(new_image_path)):
        continue
    Path(img_path).rename(new_image_path)
    print(f"[{idx + 1}/{len(draft_image_files)}] image_path : {img_path}")
    print(f"pred : {pred}")
    print(f"confidence : {confidence:.4f}")
    print("new_image_path : ", new_image_path)
    print("Done!")


Draft 이미지 개수: 0
Device: cuda
PyTorch Version: 2.9.1+cu128
CUDA Version: 12.8
cuDNN Version: 91002
Mixed Precision: Enabled


AttributeError: 'PyTorchModel' object has no attribute 'model_type'

In [ ]:
labeled_image_dir = "captcha_data/supreme_court/0/images/labeled"
pred_image_dir = "captcha_data/supreme_court/0/images/pred"
train_image_dir = "captcha_data/supreme_court/0/images/train"

labeleds = list(Path(labeled_image_dir).glob("*.png"))
preds = list(Path(pred_image_dir).glob("*.png"))
trains = list(Path(train_image_dir).glob("*.png"))

print(f"Labeled 이미지 개수: {len(labeleds)}")
print(f"Pred 이미지 개수: {len(preds)}")
print(f"Train 이미지 개수: {len(trains)}")

In [ ]:
import glob
from pathlib import Path

target_dir = Path("captcha_data/dev/0/images/pred")
image_list = glob.glob(os.path.join(target_dir, "*.png"))
filename_list = [os.path.basename(img_path) for img_path in image_list]
filename_list = [file_name for file_name in filename_list if len(file_name.split(".")[0]) != 6]
print(f"리사이즈된 이미지 개수: {len(filename_list)}")
sorted(filename_list)

In [ ]:
import os, glob, time
import logging, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

from pathlib import Path
from captchaSolver.dataclass import TrainData
import captchaSolver.engine as engine

draft_dir = Path("captcha_data/gov24/1/images/draft")
target_dir = Path("captcha_data/gov24/1/images/labeled")
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
rev = 1
image_width = 200
image_height = 50
batch_size = 32

start = time.time()

model = engine.get_captcha_model(captcha_id=captcha_id)
train_data: TrainData = model.train_data
model.train_data.rev = rev
model.train_data.image_width = image_width
model.train_data.image_height = image_height

In [ ]:
# captcha_data/gov24/1/images/pred 폴더의 이미지 크기 검사
from pathlib import Path
from PIL import Image

# 경로 설정
pred_dir = Path("captcha_data/gov24/1/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 크기가 다른 이미지 리스트
    wrong_size_images = []
    target_width = 200
    target_height = 50
    
    for img_path in image_files:
        try:
            img = Image.open(img_path)
            width, height = img.size
            
            if width != target_width or height != target_height:
                wrong_size_images.append({
                    'name': img_path.name,
                    'size': f"{width}x{height}"
                })
                
        except Exception as e:
            print(f"⚠️  오류 ({img_path.name}): {e}")
    
    # 결과 출력
    if wrong_size_images:
        print(f"\n❌ 크기가 {target_width}x{target_height}이 아닌 이미지 ({len(wrong_size_images)}개):\n")
        for idx, img_info in enumerate(wrong_size_images, 1):
            print(f"  {idx:3d}. {img_info['name']:30s} -> {img_info['size']}")
    else:
        print(f"\n✅ 모든 이미지가 {target_width}x{target_height} 크기입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 ({target_width}x{target_height}): {len(image_files) - len(wrong_size_images)}개")
    print(f"비정상: {len(wrong_size_images)}개")

In [ ]:
# captcha_data/gov24/1/images/pred 폴더의 파일명 길이 검사 (확장자 포함 10자리가 아닌 것)
from pathlib import Path

# 경로 설정
pred_dir = Path("captcha_data/gov24/0/images/pred")

if not pred_dir.exists():
    print(f"폴더가 존재하지 않습니다: {pred_dir}")
else:
    # 이미지 파일 목록
    image_files = sorted(pred_dir.glob("*.png"))
    print(f"검사할 이미지 개수: {len(image_files)}")
    print("=" * 70)
    
    # 파일명 길이가 10자리(확장자 포함)가 아닌 이미지 리스트
    wrong_name_images = []
    target_length = 10  # 예: "abc12.png" = 9자 (레이블 5자 + ".png" 4자)
    
    for img_path in image_files:
        filename = img_path.name
        name_length = len(filename)
        
        if name_length != target_length:
            wrong_name_images.append({
                'name': filename,
                'length': name_length,
                'label_length': len(img_path.stem)  # 확장자 제외한 레이블 길이
            })
    
    # 결과 출력
    if wrong_name_images:
        print(f"\n❌ 파일명 길이가 {target_length}자가 아닌 이미지 ({len(wrong_name_images)}개):\n")
        for idx, img_info in enumerate(wrong_name_images, 1):
            print(f"  {idx:3d}. {img_info['name']:40s} (길이: {img_info['length']:2d}, 레이블: {img_info['label_length']:2d}자)")
    else:
        print(f"\n✅ 모든 이미지 파일명이 {target_length}자입니다.")
    
    print("=" * 70)
    print(f"\n검사 완료!")
    print(f"전체: {len(image_files)}개")
    print(f"정상 (파일명 {target_length}자): {len(image_files) - len(wrong_name_images)}개")
    print(f"비정상: {len(wrong_name_images)}개")

In [ ]:
pred_dir = Path("captcha_data/gov24/1/images/pred")
train_dir = Path("captcha_data/gov24/1/images/train")
captcha_id = 'gov24'
rev = 1
image_width = 200
image_height = 50
batch_size = 32
model = engine.get_captcha_model(captcha_id=captcha_id)


#### 학습 데이타 섞기

In [ ]:
import captchaSolver.engine as engine

engine.redistribute_train_pred(
    image_dir="captcha_data/gov24/1/images",
    train_ratio=0.9,
    verbose=True,
)

### onnx

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ImageClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x: torch.Tensor):
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
torch_model = ImageClassifierModel()
# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
example_inputs = (torch.randn(1, 1, 32, 32),)
onnx_program = torch.onnx.export(torch_model, example_inputs, dynamo=True)

In [ ]:
onnx_program.save("image_classifier_model.onnx")

In [ ]:
import onnx

onnx_model = onnx.load("image_classifier_model.onnx")
onnx.checker.check_model(onnx_model)

In [ ]:
import onnxruntime

onnx_inputs = [tensor.numpy(force=True) for tensor in example_inputs]
print(f"Input length: {len(onnx_inputs)}")
print(f"Sample input: {onnx_inputs}")

ort_session = onnxruntime.InferenceSession(
    "./image_classifier_model.onnx", providers=["CPUExecutionProvider"]
)

onnxruntime_input = {input_arg.name: input_value for input_arg, input_value in zip(ort_session.get_inputs(), onnx_inputs)}

# ONNX Runtime returns a list of outputs
onnxruntime_outputs = ort_session.run(None, onnxruntime_input)[0]

In [ ]:
torch_outputs = torch_model(*example_inputs)

assert len(torch_outputs) == len(onnxruntime_outputs)
for torch_output, onnxruntime_output in zip(torch_outputs, onnxruntime_outputs):
    torch.testing.assert_close(torch_output, torch.tensor(onnxruntime_output))

print("PyTorch and ONNX Runtime output matched!")
print(f"Output length: {len(onnxruntime_outputs)}")
print(f"Sample output: {onnxruntime_outputs}")

In [ ]:
import torch
import onnxscript

# Opset 18 is the standard supported version as of PyTorch 2.6
from onnxscript import opset18 as op


# Create a model that uses the operator torch.ops.aten.add.Tensor
class Model(torch.nn.Module):
    def forward(self, input_x, input_y):
        return torch.ops.aten.add.Tensor(input_x, input_y)


# NOTE: The function signature (including parameter names) must match the signature of the unsupported PyTorch operator.
# https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/native_functions.yaml
# All attributes must be annotated with type hints.
def custom_aten_add(self, other, alpha: float = 1.0):
    if alpha != 1.0:
        alpha = op.CastLike(alpha, other)
        other = op.Mul(other, alpha)
    # To distinguish the custom implementation from the builtin one, we switch the order of the inputs
    return op.Add(other, self)


x = torch.tensor([1.0])
y = torch.tensor([2.0])

# Then we provide the custom implementation to the ONNX exporter as a ``custom_translation_table``.
onnx_program = torch.onnx.export(
    Model().eval(),
    (x, y),
    dynamo=True,
    custom_translation_table={
        torch.ops.aten.add.Tensor: custom_aten_add,
    },
)
# Optimize the ONNX graph to remove redundant nodes
onnx_program.optimize()

In [ ]:
print(onnx_program.model)

In [ ]:
result = onnx_program(x, y)[0]
torch.testing.assert_close(result, torch.tensor([3.0]))

In [ ]:
class GeluModel(torch.nn.Module):
    def forward(self, input_x):
        return torch.ops.aten.gelu(input_x)


# Create a namespace for the custom operator using ONNX Script
# ``com.microsoft`` is an official ONNX Runtime namespace
microsoft_op = onnxscript.values.Opset(domain="com.microsoft", version=1)

# NOTE: The function signature (including parameter names) must match the signature of the unsupported PyTorch operator.
# https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/native_functions.yaml
# NOTE: All attributes must be annotated with type hints.
# The function must be scripted using the ``@onnxscript.script()`` decorator when
# using operators from custom domains. This may be improved in future versions.
from onnxscript import FLOAT


@onnxscript.script(microsoft_op)
def custom_aten_gelu(self: FLOAT, approximate: str = "none") -> FLOAT:
    return microsoft_op.Gelu(self)


onnx_program = torch.onnx.export(
    GeluModel().eval(),
    (x,),
    dynamo=True,
    custom_translation_table={
        torch.ops.aten.gelu.default: custom_aten_gelu,
    },
)

# Optimize the ONNX graph to remove redundant nodes
onnx_program.optimize()

In [ ]:
print(onnx_program.model)

In [ ]:
result = onnx_program(x)[0]
torch.testing.assert_close(result, torch.ops.aten.gelu(x))

In [ ]:
# Define and use the operator in PyTorch
@torch.library.custom_op("mylibrary::add_and_round_op", mutates_args=())
def add_and_round_op(input: torch.Tensor) -> torch.Tensor:
    return torch.round(input + input)


@add_and_round_op.register_fake
def _add_and_round_op_fake(tensor_x):
    return torch.empty_like(tensor_x)


class AddAndRoundModel(torch.nn.Module):
    def forward(self, input):
        return add_and_round_op(input)


# Implement the custom operator in ONNX using ONNX Script
def onnx_add_and_round(input):
    return op.Round(op.Add(input, input))


onnx_program = torch.onnx.export(
    AddAndRoundModel().eval(),
    (x,),
    dynamo=True,
    custom_translation_table={
        torch.ops.mylibrary.add_and_round_op.default: onnx_add_and_round,
    },
)

# Optimize the ONNX graph to remove redundant nodes
onnx_program.optimize()
print(onnx_program)

In [ ]:
result = onnx_program(x)[0]
torch.testing.assert_close(result, add_and_round_op(x))

In [ ]:
import torch

In [ ]:
class ForwardWithControlFlowTest(torch.nn.Module):
    def forward(self, x):
        if x.sum():
            return x * 2
        return -x


class ModelWithControlFlowTest(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(3, 2),
            torch.nn.Linear(2, 1),
            ForwardWithControlFlowTest(),
        )

    def forward(self, x):
        out = self.mlp(x)
        return out


model = ModelWithControlFlowTest()

In [ ]:
x = torch.randn(3)
model(x)

try:
    torch.export.export(model, (x,), strict=False)
    raise AssertionError("This export should failed unless PyTorch now supports this model.")
except Exception as e:
    print(e)

In [ ]:
def new_forward(x):
    def identity2(x):
        return x * 2

    def neg(x):
        return -x

    return torch.cond(x.sum() > 0, identity2, neg, (x,))


print("the list of submodules")
for name, mod in model.named_modules():
    print(name, type(mod))
    if isinstance(mod, ForwardWithControlFlowTest):
        mod.forward = new_forward

In [ ]:
print(torch.export.export(model, (x,), strict=False))

In [ ]:
onnx_program = torch.onnx.export(model, (x,), dynamo=True)
print(onnx_program.model)

In [ ]:
onnx_program.optimize()
print(onnx_program.model)

### 정부 24 캡챠

In [ ]:
# 필요한 라이브러리 임포트
import requests
from pathlib import Path
from datetime import datetime
import time
from PIL import Image
from io import BytesIO

image_count = 50
url = "https://www.gov.kr/mw/captcha"
image_dir = Path("captcha_data/gov24/1/images/draft")
image_dir.mkdir(parents=True, exist_ok=True)

headers = {
    'Accept': 'image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8',
    'Accept-Language': 'ko,en;q=0.9,en-US;q=0.8',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36 Edg/142.0.0.0',
    'Connection': 'keep-alive',
}

success_count = 0
fail_count = 0

for i in range(image_count):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # response.content에서 직접 이미지 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        filename = f"{timestamp}.png"
        filepath = image_dir / filename
        
        # 이미지 형식 변환 (필요시 PNG로 저장)
        image = Image.open(BytesIO(response.content))
        image.save(filepath, format="PNG")
        
        success_count += 1
        print(f"[{i+1}/{image_count}] 저장 완료: {filename}")
        
    except Exception as e:
        fail_count += 1
        print(f"[{i+1}/{image_count}] 오류 발생: {e}")
    
    if i < image_count - 1:
        time.sleep(0.1)

print("-" * 50)
print(f"다운로드 완료! 성공: {success_count}개, 실패: {fail_count}개")
print(f"저장 위치: {image_dir.absolute()}")